In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
print(df.columns.tolist())
print(df.head())

['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   
2   3  Determine the correct option: What is the term...   
3   4  Select the most accurate option: What is Marti...   
4   5  Identify the correct statement: What is the co...   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  Accelerator-based light-ion fusion is a techni...   
2                                       Blueshifting   
3  Martin Heidegger believes that humans exist wi...   
4  Simultaneity is relative, meaning that two eve...   

                                                   B  \
0  Martin Heidegger believes that humans do not e...   
1  Accelerator-based light-ion fusion is a techni...   
2                                        Redshifting   
3  Martin Heidegger believ

In [3]:
import string
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

def clean_text(text):
    text=str(text).lower()
    text=text.translate(str.maketrans('','',string.punctuation))
    return text

df['prompt_clean']=df['prompt'].apply(clean_text)

all_words=[]
for p in df['prompt_clean']:
    all_words.extend(p.split())

vocab=set(all_words)
print("Q1",len(vocab))

859


In [4]:
row1=df[df['id']==1].iloc[0]
row1_clean=clean_text(row1['prompt'])
row1_words=row1_clean.split()
row1_filtered=[w for w in row1_words if w not in ENGLISH_STOP_WORDS]
print("Q2",row1_filtered)

['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']


In [5]:
option_cols=['A','B','C','D','E']
combined_texts=[]
for _,row in df.iterrows():
    combined_texts.append(str(row['prompt']))
    for col in option_cols:
        combined_texts.append(str(row[col]))

vectorizer=TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_texts)
vocab_size=len(vectorizer.vocabulary_)
print("Q3",vocab_size)

2762


In [6]:
row1=df[df['id']==1].iloc[0]
prompt_vec=vectorizer.transform([str(row1['prompt'])])
optionA_vec=vectorizer.transform([str(row1['A'])])
sim_q4=cosine_similarity(prompt_vec,optionA_vec)[0][0]
print("Q4",round(sim_q4,4))

0.2328


In [7]:
correct_count=0
total=len(df)

for _,row in df.iterrows():
    prompt_vec=vectorizer.transform([str(row['prompt'])])
    sims={}
    for col in option_cols:
        opt_vec=vectorizer.transform([str(row[col])])
        sims[col]=cosine_similarity(prompt_vec,opt_vec)[0][0]
    predicted=max(sims,key=sims.get)
    if predicted==row['answer']:
        correct_count+=1

accuracy_pct=correct_count/total*100
print("Q5",round(accuracy_pct,2))

13.7


In [9]:
def map_at_3(predictions,truth):
    for i,p in enumerate(predictions[:3]):
        if p==truth:
            return 1.0/(i+1)
    return 0.0
q6_score=map_at_3(['C','A','B'],'C')
print("Q6: MAP@3 score =",q6_score)

Q6: MAP@3 score = 1.0


In [10]:
q7_score=map_at_3(['D','B','E'],'B')
print("Q7: MAP@3 score =",q7_score)

Q7: MAP@3 score = 0.5


In [12]:
answer_counts=df['answer'].value_counts()
print(answer_counts)
top3_answers=answer_counts.index[:3].tolist()
print(top3_answers)

scores=[]
for _,row in df.iterrows():
    truth=row['answer']
    scores.append(map_at_3(top3_answers,truth))
map3_baseline=np.mean(scores)
print("Q8",round(map3_baseline,4))

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
['B', 'C', 'A']
Q8 0.4212


In [ ]:
scores_tfidf=[]
for _,row in df.iterrows():
    truth=row['answer']
    prompt_vec=vectorizer.transform([str(row['prompt'])])
    sims={}
    for col in option_cols:
        opt_vec=vectorizer.transform([str(row[col])])
        sims[col]=cosine_similarity(prompt_vec,opt_vec)[0][0]
    ranked=sorted(sims.keys(),key=lambda k:sims[k],reverse=True)
    top3=ranked[:3]
    scores_tfidf.append(map_at_3(top3,truth))

map3_tfidf=np.mean(scores_tfidf)
print("Q9",round(map3_tfidf,4))